In [ ]:
import subprocess, sys, os, time, glob, json, shutil, hashlib, threading, zipfile
T_NB = time.time()
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode
sh(f"{sys.executable} -m pip install -q 'transformers==5.20.2' 'peft==0.21.2' 'trl==0.29.1' accelerate bitsandbytes pycocoevalcap sacrebleu 2>&1 | tail -3")
sh(f"{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1")
import transformers, peft, torch
print(transformers.__version__, peft.__version__, torch.__version__, torch.cuda.device_count(), flush=True)
assert (transformers.__version__, peft.__version__) == ("5.20.2", "0.21.2"), "DUNG: lech phien ban"
assert torch.cuda.device_count() == 2, "DUNG: can 2 GPU T4"


In [ ]:
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
D = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/sa3_dev.py", recursive=True) if "/notebooks/" not in p]
assert len(D) == 1, D; D = D[0]
for f, h in {'sa3_dev.py': 'd76305565a58b074a0623e8d9b73edad', 'sa3_grpo.py': 'c93bb07f5e761b5448d85ef327c9d5c7', 'grpo_luoi.py': '92d0a9bc6b8cae8a480aef08f99fb9d8', 'grpo_spice.py': '8489108147c8d1ed80c3dc67c8ad3d3f', 'build_branch_data.py': '619e63e123a6dbf60086e65ee94a3912', 'sa3_reward_screen.py': 'f6637048ed5dce6db051b4fabf15833d', 'sa3_action_verify.py': '672e2b6ff04155a7214e8880509089a2', 'h4_spice_tuple_7_11.py': 'd5f13d93fb84235f60c4bd2e558a54a5', 'dev_v3.jsonl': '7dba2a79e9da0759f10fcfd5a7e8e2eb', 'recs_tap.jsonl': '24030d570098b54c2e1ccd22d801b08d', 'ocr_dev.jsonl': 'f527cc537a1c7075a091ff4595fe4bc5', 'ad/sa_ck500/adapter_model.safetensors': '406c6931a2072304c22af8fdf2e175a7'}.items():
    assert md5(f"{D}/{f}") == h, f"DUNG: {f} lech ban may nha"
for f in ('sa3_dev.py', 'sa3_grpo.py', 'grpo_luoi.py', 'grpo_spice.py', 'build_branch_data.py', 'sa3_reward_screen.py', 'sa3_action_verify.py', 'h4_spice_tuple_7_11.py'):
    shutil.copy(f"{D}/{f}", f"{W}/{f}")
WS = f"{W}/pk/thesis"; shutil.copytree(f"{D}/thesis", WS, dirs_exist_ok=True)
VD = f"{W}/data"; os.makedirs(VD, exist_ok=True)
shutil.copy(f"{D}/recs_tap.jsonl", f"{VD}/recs_tap.jsonl")
if not os.path.lexists(f"{VD}/images"): os.symlink(f"{D}/images", f"{VD}/images")
ENV = {**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def py(*a, gpu=0):
    r = subprocess.run(["python", *a], cwd=W, capture_output=True, text=True, env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)})
    print((r.stdout + r.stderr)[-2500:], flush=True); return r.returncode
S1M, SFTM = "/tmp/s1_merged", "/tmp/sft_cont_merged"
assert py("grpo_spice.py", "--merge", "--bundle", D, "--adapter", f"{D}/adapter_s1_seed101", "--merged", S1M) == 0, "DUNG: hoa S1"
assert py("sa3_grpo.py", "--hoa", "--data", D, "--merged-s1", S1M, "--merged", SFTM) == 0, "DUNG: hoa SFT-cont"
shutil.rmtree(S1M, ignore_errors=True)
assert py("sa3_grpo.py", "--kiem-hoa", "--data", D, "--merged", SFTM) == 0, "DUNG: kiem hoa"
from huggingface_hub import snapshot_download
print("UGround:", snapshot_download("osunlp/UGround-V1-2B"), flush=True)
sys.path.insert(0, f"{WS}/harness")
import a11y_inventory as A11Y, score_run as SR
recs = [json.loads(l) for l in open(f"{VD}/recs_tap.jsonl")]
names = set(A11Y._zip().namelist())
co = sum(A11Y.key_for(f"episode_{d['episode_id']}_screenshot_{d['step_id']}.png") in names for d in recs)
rong = sum(len(SR.buttons_of(d)) == 0 for d in recs)
print(f"buoc cham {len(recs)} · cay tro nang {co}/1338 · rong nut {rong}", flush=True)
assert len(recs) == 1338 and co == 1338 and rong <= 5, "DUNG: thieu cay tro nang"
print("✅ moi truong + hoa + kiem hoa + UGround", flush=True)


In [ ]:
MODELS = ['sftcont', 'sa_ck500']
def gen_cmd(mo, out, n=0):
    c = ["python", "-u", "sa3_dev.py", "--data", D, "--merged", SFTM, "--out", out]
    if mo != "sftcont": c += ["--adapter", f"{D}/ad/{mo}"]
    return c + (["--n", str(n)] if n else [])
score = lambda mo, out: ["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground", "--preds", f"{W}/dev_{mo}.jsonl",
                         "--data-root", VD, "--recs-file", "recs_tap.jsonl", "--out", out]
# thử 3 bước mỗi mô hình (sinh) + 3 bước chấm
for g, mo in enumerate(MODELS):
    assert py(*gen_cmd(mo, f"{W}/thu_{mo}.jsonl", 3)[1:], gpu=g) == 0, f"DUNG: thu sinh {mo}"
    T = [json.loads(l) for l in open(f"{W}/thu_{mo}.jsonl")]
    assert len(T) == 3 and all(t["pred"] for t in T), f"DUNG: thu {mo} sai"
    print("✅ thu", mo, [(t["gold"], t["pred"]) for t in T], flush=True)
CAT = 3600 * 10.5
RC = {}
def chay(g, mo):
    lg = open(f"{W}/{mo}.log", "a")
    for buoc, cmd, cwd in (("sinh", gen_cmd(mo, f"{W}/dev_{mo}.jsonl"), W), ("cham", score(mo, f"{W}/score/score_dev_{mo}.json"), WS)):
        os.makedirs(f"{W}/score", exist_ok=True)
        q = subprocess.Popen(cmd, cwd=cwd, stdout=lg, stderr=subprocess.STDOUT, env={**ENV, "CUDA_VISIBLE_DEVICES": str(g)}, start_new_session=True)
        while q.poll() is None:
            time.sleep(60)
            if time.time() - T_NB > CAT:
                q.terminate(); q.wait(); RC[mo] = f"het gio o {buoc}"; return
        if q.returncode != 0:
            RC[mo] = f"loi {buoc} {q.returncode}"; return
        if buoc == "sinh":
            n = sum(1 for _ in open(f"{W}/dev_{mo}.jsonl")); assert n == 2223, n
    RC[mo] = 0
TH = [threading.Thread(target=chay, args=(g, mo)) for g, mo in enumerate(MODELS)]
[t.start() for t in TH]
while any(t.is_alive() for t in TH):
    time.sleep(120)
    for mo in MODELS:
        L = [l.strip() for l in open(f"{W}/{mo}.log", errors="ignore") if l.strip()] if os.path.exists(f"{W}/{mo}.log") else []
        print(f"{(time.time()-T_NB)/3600:5.2f} h · {mo} · {L[-1][:140] if L else 'chua co dong'}", flush=True)
print("ket qua", RC, flush=True)
for mo in MODELS:
    if RC.get(mo) != 0: print(open(f"{W}/{mo}.log", errors="ignore").read()[-2500:], flush=True)
    p = f"{W}/score/score_dev_{mo}.json"
    if os.path.exists(p): print(mo, {k: v for k, v in json.load(open(p)).items() if not isinstance(v, (list, dict))}, flush=True)


In [ ]:
with zipfile.ZipFile(f"{W}/sa3_dev_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{W}/*.jsonl") + glob.glob(f"{W}/*.log") + glob.glob(f"{W}/score/*"):
        z.write(p, os.path.relpath(p, W))
print("zip", os.path.getsize(f"{W}/sa3_dev_out.zip") // 1024, "KB", flush=True)
shutil.rmtree(SFTM, ignore_errors=True)
